# Импорт библиотек, загрузка датасета и его осмотр.

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mlp

In [2]:
df = pd.read_csv('/content/drive/MyDrive/datasets/topic_small.csv') # датасет в гугл драйве

Смотрим как выглядит первые 5 объектов

In [3]:
df.loc[df['topic'].isna(),'topic'].sum() #есть ли наны в целевой

0

In [4]:
df.head()

,Unnamed: 0,url,title,text,topic,tags,date
0,390718,https://lenta.ru/news/2011/09/26/cost1/,Расходы Великобритании на операцию в Ливии нед...,Расходы правительства Великобритании на военну...,Мир,Все,2011/09/26
1,661864,https://lenta.ru/news/2017/05/19/arthouse/,Мединский предрек превращение российского кино...,Если не оказывать отечественному кинематографу...,Культпросвет,События,2017/05/19
2,139745,https://lenta.ru/news/2005/12/27/offer/,Украина предложила 80 долларов за тысячу кубом...,Украина готова платить 80 долларов за тысячу к...,Бывший СССР,Все,2005/12/27
3,655163,https://lenta.ru/news/2017/04/08/badrussian/,Маккейн назвал Россию и президента Сирии «один...,Член сената Конгресса США республиканец Джон М...,Мир,Политика,2017/04/08
4,338861,https://lenta.ru/news/2010/07/23/mansion/,Гильермо дель Торо сделает фильм из диснеевско...,Гильермо дель Торо сделает фильм из диснеевско...,Культура,Все,2010/07/23


In [5]:
df.shape

(100000, 7)

Сколько классов и какие из них можно было бы убрать

In [6]:
df['topic'].value_counts()

,count
topic,
Россия,20082
Мир,17120
Экономика,9929
Спорт,8042
Наука и техника,6665
Бывший СССР,6601
Культура,6568
Интернет и СМИ,5563
Из жизни,3438


Все, что меньше путешествий по количеству можно убрать, так как их слишком мало, смысла особо для модели не имеют

# Корректировка датасета

## Обрезание малоинформативных топиков

In [7]:
topics = df['topic'].value_counts()

In [8]:
topics_yes = topics[topics>=500].index # оставляем топики где количество статей больше 500

In [9]:
topics_yes

Index(['Россия', 'Мир', 'Экономика', 'Спорт', 'Наука и техника', 'Бывший СССР',
       'Культура', 'Интернет и СМИ', 'Из жизни', 'Дом', 'Силовые структуры',
       'Ценности', 'Бизнес', 'Путешествия'],
      dtype='object', name='topic')

In [10]:
df = df[df['topic'].isin(topics_yes)].reset_index(drop=True)

In [11]:
df['topic'].value_counts()

,count
topic,
Россия,20082
Мир,17120
Экономика,9929
Спорт,8042
Наука и техника,6665
Бывший СССР,6601
Культура,6568
Интернет и СМИ,5563
Из жизни,3438


In [12]:
df.shape

(91957, 7)

## Обработка текстов статей

In [13]:
df['text'].iloc[0] # смотрим как выглядит текст одной из статей, чтобы понять как обработать

'Расходы правительства Великобритании на военную операцию в Ливии могут достигнуть 1,75 миллиарда фунтов, сообщает The Guardian со ссылкой на эксперта авторитетного издания Defence Analysis. Согласно расчетам эксперта Фрэнсиса Тусы (Francis Tusa), правительство могло в семь раз недооценить возможные расходы на участие военно-воздушных сил в бомбардировках сил Муаммара Каддафи. Для расчета расходов на бомбардировки Ливии Туса опирался на данные, предоставленные членами британского парламента и представителями военно-воздушных сил. При своих подсчетах он использовал две разные методики: в первом случае полученный результат колебался между 1,38 и 1,58 миллиарда фунтов, во втором случае разброс оказался еще больше - от 850 миллионов до 1,75 миллиарда фунтов. При этом Туса подчеркнул, что в своих расчетах он не учитывал последние вылеты королевских ВВС в Ливии в сентябре. Таким образом, по окончании операции итоговая сумма может существенно вырасти. Для сравнения, США потратили на операцию 

In [14]:
import re

In [15]:
def text_clean(text):
  text = str(text).lower() # нижний регистр
  # text = re.sub(r'https:\S+|www\S+','URLTOKEN',text)
  text = re.sub(r'\d+',' numtoken ',text)
  text = re.sub(r'[^а-яёa-z\s]',' ',text)# убираем все другие символы кроме кириллицы и латиницы
  text = re.sub(r'\s+',' ',text)
  return text

In [16]:
df['text_clean'] = df['text'].apply(text_clean)

In [17]:
df['text_clean'].iloc[0]

'расходы правительства великобритании на военную операцию в ливии могут достигнуть numtoken numtoken миллиарда фунтов сообщает the guardian со ссылкой на эксперта авторитетного издания defence analysis согласно расчетам эксперта фрэнсиса тусы francis tusa правительство могло в семь раз недооценить возможные расходы на участие военно воздушных сил в бомбардировках сил муаммара каддафи для расчета расходов на бомбардировки ливии туса опирался на данные предоставленные членами британского парламента и представителями военно воздушных сил при своих подсчетах он использовал две разные методики в первом случае полученный результат колебался между numtoken numtoken и numtoken numtoken миллиарда фунтов во втором случае разброс оказался еще больше от numtoken миллионов до numtoken numtoken миллиарда фунтов при этом туса подчеркнул что в своих расчетах он не учитывал последние вылеты королевских ввс в ливии в сентябре таким образом по окончании операции итоговая сумма может существенно вырасти д

# Лемматизация текста

In [18]:
!pip install pymorphy3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 67.9 MB/s eta 0:00:00


In [19]:
import pymorphy3

In [20]:
from collections import Counter
all_words = ' '.join(df['text_clean']).split() # все слова
unique_words = set(all_words)# уникальные
print(len(unique_words))  # вывод сколько уникальных слов

408206


In [21]:
morph = pymorphy3.MorphAnalyzer()
def lemmatize(text):
    words = text.split()
    lemmas = [morph.parse(word)[0].normal_form for word in words]
    return ' '.join(lemmas)
# лемматизация без кэширования (словарик уникальных слов не нужен)

In [22]:
lemma_dict = {word: morph.parse(word)[0].normal_form for word in unique_words}
def lemmatize_cache(text):
  return ' '.join(lemma_dict.get(word,word) for word in text.split())
df['text_lemma'] = df['text_clean'].apply(lemmatize_cache)
# лемматизация с кэшированием

In [23]:
df.loc[df['topic'].isna(),['topic','text_lemma']]

,topic,text_lemma


In [24]:
df.shape

(91957, 9)

# Деление выборки и модель

In [25]:
from sklearn.model_selection import train_test_split

In [26]:
X = df['text_lemma']
y = df['topic']
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.1,random_state=42,stratify=y)

## Векторизация

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000, min_df=3)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [28]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

## Метрики

In [29]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_val_tfidf)
print(classification_report(y_val, y_pred))

                   precision    recall  f1-score   support

           Бизнес       0.68      0.18      0.29        93
      Бывший СССР       0.83      0.82      0.82       660
              Дом       0.87      0.75      0.81       273
         Из жизни       0.70      0.62      0.65       344
   Интернет и СМИ       0.79      0.73      0.76       556
         Культура       0.87      0.88      0.87       657
              Мир       0.79      0.84      0.82      1712
  Наука и техника       0.83      0.87      0.85       667
      Путешествия       0.90      0.54      0.68        85
           Россия       0.77      0.83      0.80      2008
Силовые структуры       0.70      0.41      0.52       250
            Спорт       0.96      0.97      0.97       804
         Ценности       1.00      0.74      0.85        94
        Экономика       0.83      0.86      0.85       993

         accuracy                           0.82      9196
        macro avg       0.82      0.72      0.75      

Здесь сделал сбалансированную модель, то есть она будет уделять больше внимания более редким топикам

In [30]:
model_balanced = LogisticRegression(max_iter=1000, n_jobs=-1, class_weight='balanced')
model_balanced.fit(X_train_tfidf, y_train)

y_pred_balanced = model_balanced.predict(X_val_tfidf)
print(classification_report(y_val, y_pred_balanced))

                   precision    recall  f1-score   support

           Бизнес       0.38      0.69      0.49        93
      Бывший СССР       0.79      0.88      0.83       660
              Дом       0.71      0.85      0.78       273
         Из жизни       0.56      0.77      0.65       344
   Интернет и СМИ       0.75      0.77      0.76       556
         Культура       0.88      0.88      0.88       657
              Мир       0.84      0.78      0.81      1712
  Наука и техника       0.82      0.85      0.84       667
      Путешествия       0.55      0.85      0.66        85
           Россия       0.86      0.69      0.77      2008
Силовые структуры       0.46      0.75      0.57       250
            Спорт       0.96      0.97      0.97       804
         Ценности       0.87      0.89      0.88        94
        Экономика       0.87      0.82      0.84       993

         accuracy                           0.80      9196
        macro avg       0.74      0.82      0.77      

# Дообучение модели ruBERT-tiny2 для классификации

In [31]:
df_bert = df[['text','topic']].copy()

In [32]:
df_sample, _ = train_test_split(df_bert, train_size = 18000, stratify = df_bert['topic'],random_state = 42)
df_train,df_val = train_test_split(df_sample,test_size=0.1,stratify=df_sample['topic'],random_state= 42)
print(df_train.shape, ' ',df_val.shape)

(16200, 2)   (1800, 2)


In [33]:
df_sample['topic'].value_counts() #смотрим сохранились ли отношения классов

,count
topic,
Россия,3931
Мир,3351
Экономика,1943
Спорт,1574
Наука и техника,1305
Бывший СССР,1292
Культура,1286
Интернет и СМИ,1089
Из жизни,673


In [34]:
from sklearn.preprocessing import LabelEncoder

## Кодировка

In [35]:
enc = LabelEncoder() # кодирование категориальных данных
df_train['label'] = enc.fit_transform(df_train['topic'])
df_val['label'] = enc.fit_transform(df_val['topic'])

In [36]:
enc.classes_

array(['Бизнес', 'Бывший СССР', 'Дом', 'Из жизни', 'Интернет и СМИ',
       'Культура', 'Мир', 'Наука и техника', 'Путешествия', 'Россия',
       'Силовые структуры', 'Спорт', 'Ценности', 'Экономика'],
      dtype=object)

## Трансформер

In [37]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [38]:
model_name = 'cointegrated/rubert-tiny2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_bert = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels = len(enc.classes_))

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
#проверка на наны
print(df_train['text'].isnull().sum())
print(df_val['text'].isnull().sum())

# Проверим типы данных в столбце
print(df_train['text'].apply(type).value_counts())
##print(df_val['text'].apply(type).value_counts())

1
0
text
<class 'str'>      16199
<class 'float'>        1
Name: count, dtype: int64


In [40]:
df_train.loc[df_train['text'].isnull()]

,text,topic,label
78371,NaN,Культура,5


In [41]:
df_train.loc[78371]

,78371
text,NaN
topic,Культура
label,5


In [42]:
df_train = df_train.drop(78371)

### Создание необходимых функций

In [43]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256) #функция токенизации

from datasets import Dataset

train_dataset = Dataset.from_pandas(df_train[['text', 'label']])
val_dataset = Dataset.from_pandas(df_val[['text', 'label']])

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16199 [00:00<?, ? examples/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

In [44]:
from sklearn.metrics import accuracy_score,f1_score

In [45]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=1)
  acc = accuracy_score(labels,predictions)
  f1_macro = f1_score(labels,predictions,average = 'macro')
  f1_weighted = f1_score(labels,predictions,average = 'weighted')
  return{'accuracy =':acc, 'f1 score(macro) =':f1_macro,'f1 score(weighted) =':f1_weighted
  } #функция вычисления метрик

In [46]:
from transformers import TrainingArguments, Trainer

In [47]:
training_args = TrainingArguments(output_dir='./bert_results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1 score(macro) =',
    logging_steps=50,
    report_to='none'
)

In [48]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # использование серверов колаба

In [49]:
model_bert.to(device)# переключаем модель на колаб

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, 

In [50]:
trainer = Trainer(model = model_bert,args = training_args,train_dataset=train_dataset,eval_dataset=val_dataset,compute_metrics=compute_metrics)
trainer.train() # обучение

Epoch,Training Loss,Validation Loss,Accuracy =,F1 score(macro) =,F1 score(weighted) =
1,1.091863,1.049870,0.716667,0.444061,0.672439
2,0.848832,0.811953,0.775556,0.578174,0.750831
3,0.706748,0.731396,0.778889,0.599275,0.758340
4,0.614809,0.704244,0.779444,0.616320,0.762237
5,0.578759,0.694953,0.788889,0.636568,0.773341


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5065, training_loss=0.8593321511211188, metrics={'train_runtime': 291.5136, 'train_samples_per_second': 277.843, 'train_steps_per_second': 17.375, 'total_flos': 299104224660480.0, 'train_loss': 0.8593321511211188, 'epoch': 5.0})

In [51]:
rez = trainer.evaluate() # лучший результат

Training Loss,Validation Loss,Epoch,Accuracy =,F1 score(macro) =,F1 score(weighted) =
0.578759,0.694953,5,0.788889,0.636568,0.773341


## Результаты классического и глубокого подходов на тренировочных данных

In [52]:
print(classification_report(y_val, y_pred_balanced))#Классика

                   precision    recall  f1-score   support

           Бизнес       0.38      0.69      0.49        93
      Бывший СССР       0.79      0.88      0.83       660
              Дом       0.71      0.85      0.78       273
         Из жизни       0.56      0.77      0.65       344
   Интернет и СМИ       0.75      0.77      0.76       556
         Культура       0.88      0.88      0.88       657
              Мир       0.84      0.78      0.81      1712
  Наука и техника       0.82      0.85      0.84       667
      Путешествия       0.55      0.85      0.66        85
           Россия       0.86      0.69      0.77      2008
Силовые структуры       0.46      0.75      0.57       250
            Спорт       0.96      0.97      0.97       804
         Ценности       0.87      0.89      0.88        94
        Экономика       0.87      0.82      0.84       993

         accuracy                           0.80      9196
        macro avg       0.74      0.82      0.77      

In [53]:
print(rez)#Трансформер

{'eval_loss': 0.6949528455734253, 'eval_accuracy =': 0.7888888888888889, 'eval_f1 score(macro) =': 0.6365680953014117, 'eval_f1 score(weighted) =': 0.7733406031244829}


# Дообученная модель и catboost на новых данных

## Загрузка полного датасета, из которого был получен тренировочный датасет

In [54]:
testdf = pd.read_csv('https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.1/lenta-ru-news.csv.bz2',compression = 'bz2')

/tmp/ipykernel_2252/3306086287.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  testdf = pd.read_csv('https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.1/lenta-ru-news.csv.bz2',compression = 'bz2')


In [55]:
testdf.describe()

,url,title,text,topic,tags,date
count,800975,800975,800970,738973,773756,800975
unique,800964,797832,800037,23,94,7393
top,https://lenta.ru/news/2004/06/21/hostage/,В Москве объявлено штормовое предупреждение,"РИА ""Новости""",Россия,Все,2019/12/05
freq,2,21,291,160445,453762,284


In [56]:
testdf = testdf.sample(100000,random_state = 42).reset_index(drop=True) #Обрезаем

In [57]:
testdf = testdf.dropna(subset=['topic'])

In [58]:
testdf['topic']

,topic
0,Культура
1,Наука и техника
2,Мир
3,Интернет и СМИ
4,Россия
...,...
99994,Экономика
99995,Россия
99996,Интернет и СМИ
99997,Россия


In [59]:
testdf['topic'].isna().sum()

np.int64(0)

## Предобработка текстов, так же как и на тренировочных

In [60]:
testdf['text_clean'] = testdf['text'].apply(text_clean)

In [61]:
new_all_words = ' '.join(testdf['text_clean']).split() # все слова нового датасета
new_unique_words = set(new_all_words)# уникальные
print(len(unique_words))

408206


In [62]:
testdf['text_lemma'] = testdf['text_clean'].apply(lemmatize_cache)

## Векторизация и результат логистической регрессии

In [63]:
X_new = vectorizer.transform(testdf['text_lemma'])
testdf['predicted_topic'] = model.predict(X_new)

In [64]:
print(classification_report(testdf['topic'],testdf['predicted_topic']))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                   precision    recall  f1-score   support

   69-я параллель       0.00      0.00      0.00       152
       Библиотека       0.00      0.00      0.00         8
           Бизнес       0.71      0.23      0.35       979
      Бывший СССР       0.84      0.84      0.84      6772
              Дом       0.87      0.78      0.83      2661
         Из жизни       0.71      0.59      0.64      3503
   Интернет и СМИ       0.79      0.73      0.76      5591
             Крым       0.00      0.00      0.00        84
    Культпросвет        0.00      0.00      0.00        46
         Культура       0.87      0.89      0.88      6662
          Легпром       0.00      0.00      0.00        17
       МедНовости       0.00      0.00      0.00         1
              Мир       0.80      0.85      0.82     16942
  Наука и техника       0.83      0.85      0.84      6711
      Путешествия       0.84      0.58      0.68       795
           Россия       0.78      0.85      0.81     19

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Применение дообученой модели(через pipeline) и результат

In [65]:
from transformers import pipeline

In [66]:
id2label = {i: cls for i, cls in enumerate(enc.classes_)}
label2id = {cls: i for i, cls in enumerate(enc.classes_)}

model_bert.config.id2label = id2label
model_bert.config.label2id = label2id

In [67]:
classifier = pipeline('text-classification', model=model_bert, tokenizer=tokenizer,
                       device=0 if torch.cuda.is_available() else -1)

In [68]:
results = classifier(testdf['text'].tolist(), truncation=True, max_length=256, batch_size=32)
testdf['predicted_topic'] = [r['label'] for r in results]

In [69]:
testdf['predicted_topic']

,predicted_topic
0,Культура
1,Наука и техника
2,Мир
3,Интернет и СМИ
4,Россия
...,...
99994,Экономика
99995,Россия
99996,Интернет и СМИ
99997,Экономика


In [70]:
print(classification_report(testdf['topic'],testdf['predicted_topic']))

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                   precision    recall  f1-score   support

   69-я параллель       0.00      0.00      0.00       152
       Библиотека       0.00      0.00      0.00         8
           Бизнес       0.46      0.04      0.07       979
      Бывший СССР       0.77      0.86      0.81      6772
              Дом       0.70      0.78      0.74      2661
         Из жизни       0.62      0.44      0.52      3503
   Интернет и СМИ       0.71      0.71      0.71      5591
             Крым       0.00      0.00      0.00        84
    Культпросвет        0.00      0.00      0.00        46
         Культура       0.79      0.90      0.84      6662
          Легпром       0.00      0.00      0.00        17
       МедНовости       0.00      0.00      0.00         1
              Мир       0.80      0.82      0.81     16942
  Наука и техника       0.79      0.84      0.81      6711
      Путешествия       0.91      0.09      0.16       795
           Россия       0.77      0.81      0.79     19

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Итоги

Посмотрим на результаты. Усредненные не взвешенные оценки такие себе, произошло это из-за того что оба датасета я обрезал неаккуратно и грубо, через sample. И поэтому в тренировочной и тестовых выборках разное количество классов(плюс в тренировочном я дополнительно урезал малоинформативные топики). Собственно поэтому и усредненные значения такие низкие. Однако если смотреть на классы по отдельности, то ситуация лучше. С информативные классы модель и выцепляет хорошо(большой recall) и классифицирует тоже хорошо(большой precision). С редкими ситуация другая. Ловит модель плохо(низкий recall), но классифицирует получше(средний - большой precision). Но вот взвешенные оценки в целом очень даже хороши.

В целом, исправив датасеты и научив на них модель, можно получить значительно лучший результат. Также стоит учитывать, что я использовал модель с маленьким числом параметров(rubert-tiny) и обрезал выборку обучения для берта, это тоже повлияло на результат.

Итог таков: в этой ситуации классический подход оказался СИЛЬНО лучше чем нейросетевой подход. Классика:быстрее, проще в написании кода и архитектуре в целом, меньше по ресурсным затратам. Я не утверждаю, что классика точнее, так как изначальные данные не совсем равны как я описал выше, но и суть сравнения не в точности(нейросетевой подход при остальных равных будет в любом случае будет точнее), а в ответе на вопрос: стоит ли точность всех затрат ресурсов и времени.